In [1]:
import json
import os


In [2]:
# topologyType is the topology to skim for.
# 0: direct photons,
# 1: one planar refl. (selected by planarMirror)
# 2: one sph + one planar (selected by planarMirror and sphericalMirror)
# 3: one sph + two planar reflections (selected by planarMirror{1,2} and sphericalMirror)
# 4: two planar reflections (selected by planarMirror{1,2})
# 5: select tracks with a matching cluster for global alignment

# It would be more efficient to instead define some set of topologies that we want to keep
# and supply them to a script that checks for them all at once, then requiring that we only
# read each hipo file once. However, I found this approach easier for now.

jsonconfig = {
    "applyDISCut": True, # good electron + hadron (TODO: should remove, rename, or make an actual DIS cut)
    "applyRichOneCut": True, # one track in the RICH
    "topologyType": 2, # Topology to select. 0: direct photons, 1: one planar refl. 2:
    "aerolayer": 203, # aerogel layer 
    "planarMirror": 12,
    "planarMirror1": 0,
    "planarMirror2": 0,
    "sphericalMirror": 25,
    "minPhotons": 2, # Minimum photons of the desired topology to keep an event
    "maxev": 50000, # Maximum number of events to keep
    "validPIDs": [-211], # EB pid selection
    "maxPerTile": 1000, # maximum number of events kept per tile
    "sector":4 # RICH sector
}
def createNewSkimJson(topo,layer,planar,spherical,photons,maxev,outname,pids,planar1,planar2,maxPerTile,sector,minP=1.25):
    # Create a modified copy with new values
    new_config = jsonconfig.copy()
    new_config.update({
        "topologyType":topo,
        "aerolayer": layer,       
        "planarMirror": planar,     
        "planarMirror1": planar1,     
        "planarMirror2": planar2,     
        "sphericalMirror": spherical,
        "minPhotons":photons,
        "maxev":maxev,        
        "validPIDs":pids,
        "maxPerTile":maxPerTile,
        "sector":sector,
        "minP":minP
    })

    # Save to file
    with open(outname, "w") as f:
        json.dump(new_config, f, indent=4)

    print(f"Wrote {outname}")
    return


In [5]:
# pion skims
# RGK data (outbending only)
# with minimum p cut
maxev = 20000
max_pref = 20
max_per_tile = 1000
minP = {201:3.0, 202: 2.0, 203: 1.5}
outdir = "/hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/"
aide_home = "/hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/"
datadir = "/cwork/cmp115/AIDE/CLAS12-RICH-Align/dataset/"
commands_file = outdir+"skim_topo_RGK_19706_19765_PIONS_command_20k_alltopo_sector4_momCut.slurm"
input_file = "inputs_19706_19765_combined.txt" # file with list of hipo files to read (full path)
run_string = "RGK_19706_19765_combined"
mirrors_row1 = [21,25,22]
mirrors_row2 = [28,29,30,23,26,27,24]
sector = 4
for pid, pidname in zip([211,-211],["PIP","PIM"]):
    for s in mirrors_row1:
        for p in [12,13]:
            for l in [202,203]:
                pmin = minP[l]
                json_name = f"skim_for_sector{sector}_{max_pref}k_l{l}_s{s}_p{p}_{pidname}_minp.json"
                createNewSkimJson(2,l,p,s,2,maxev,outdir+json_name,[pid],0,0,max_per_tile,sector,pmin)
                with open(commands_file, "a") as f:
                    f.write(
                        f"{aide_home}/Clas12RichUtils/runSkimCommand.sh "
                        f"{aide_home}/Clas12RichUtils/RICH-skim-onetop {datadir}/skim_sector{sector}_{run_string}_{pidname}_{max_pref}k_l{l}_s{s}_p{p}_minp{pmin}.hipo @{aide_home}/Clas12RichUtils/{input_file} "
                        f"--config {aide_home}/Clas12RichUtils/{json_name}\n"
                    )

    for s in mirrors_row2:
        for p in [12,13]:
            for l in [202,203]:
                pmin = minP[l]
                json_name = f"skim_for_sector{sector}_{max_pref}k_l{l}_s{s}_p{p}_{pidname}_minp.json"
                createNewSkimJson(2,l,p,s,2,maxev,outdir+json_name,[pid],0,0,max_per_tile,sector,pmin)
                with open(commands_file, "a") as f:
                    f.write(
                        f"{aide_home}/Clas12RichUtils/runSkimCommand.sh "
                        f"{aide_home}/Clas12RichUtils/RICH-skim-onetop {datadir}/skim_sector{sector}_{run_string}_{pidname}_{max_pref}k_l{l}_s{s}_p{p}_minp{pmin}.hipo @{aide_home}/Clas12RichUtils/{input_file} "
                        f"--config {aide_home}/Clas12RichUtils/{json_name}\n"
                    )

    #topo,layer,planar,spherical,photons,maxev,outname
    for l in [201,202]:
        pmin = minP[l]
        json_name = f"skim_for_sector{sector}_{max_pref}k_l{l}_direct_{pidname}_minp.json"
        createNewSkimJson(0,l,-1,-1,3,maxev,outdir+json_name,[pid],0,0,max_per_tile,sector,pmin)
        with open(commands_file, "a") as f:
            f.write(
                f"{aide_home}/Clas12RichUtils/runSkimCommand.sh "
                f"{aide_home}/Clas12RichUtils/RICH-skim-onetop {datadir}/skim_sector{sector}_{run_string}_{pidname}_{max_pref}k_l{l}_direct_minp{pmin}.hipo @{aide_home}/Clas12RichUtils/{input_file} "
                f"--config {aide_home}/Clas12RichUtils/{json_name}\n"
            )


    mirrors_planar = [11,14,15,16,17]
    #createNewSkimJson(topo,layer,planar,spherical,photons,maxev,outname,pids):
    for p in mirrors_planar:
        for l in [201,202,203]:
            pmin = minP[l]
            json_name = f"skim_for_sector{sector}_{max_pref}k_l{l}_s0_p{p}_{pidname}_minp.json"
            createNewSkimJson(1,l,p,0,2,maxev,outdir+json_name,[pid],0,0,max_per_tile,sector,pmin)
            with open(commands_file, "a") as f:
                f.write(
                        f"{aide_home}/Clas12RichUtils/runSkimCommand.sh "
                        f"{aide_home}/Clas12RichUtils/RICH-skim-onetop {datadir}/skim_sector{sector}_{run_string}_{pidname}_{max_pref}k_l{l}_s0_p{p}_minp{pmin}.hipo @{aide_home}/Clas12RichUtils/{input_file} "
                        f"--config {aide_home}/Clas12RichUtils/{json_name}\n"
                )            

Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l202_s21_p12_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l203_s21_p12_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l202_s21_p13_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l203_s21_p13_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l202_s25_p12_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l203_s25_p12_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l202_s25_p13_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp115/AIDE/CLAS12-RICH-Align/Clas12RichUtils/skim_for_sector4_20k_l203_s25_p13_PIP_minp.json
Wrote /hpc/group/vossenlab/cmp11